In [ ]:
from diffusers import DDIMScheduler
import torch
import os
import torchvision
from rgbx.rgb2x.load_image import load_exr_image, load_ldr_image
from rgbx.rgb2x.pipeline_rgb2x import StableDiffusionAOVMatEstPipeline
import lpips
from PIL import Image
import torchvision.transforms as T
from glob import glob
import pandas as pd

from material_aware_flare.eval import calculate_metrics

device = 'cuda'

pipe = StableDiffusionAOVMatEstPipeline.from_pretrained(
    "zheng95z/rgb-to-x",
    torch_dtype=torch.float16,
    cache_dir=os.path.join('./', "model_cache_rgb_x"),
).to(device)
pipe.scheduler = DDIMScheduler.from_config(
    pipe.scheduler.config, rescale_betas_zero_snr=True, timestep_spacing="trailing"
)
pipe.to(device)

In [ ]:
def rgbx_pipeline(
    pipe,
    filename,
    num_inference_steps,
    seed,
    required_aovs=["albedo", "normal", "roughness", "metallic", "irradiance"]
):
  generator = torch.Generator(device=device).manual_seed(seed)
  photo = load_ldr_image(filename, from_srgb=True).to(device)
  # Check if the width and height are multiples of 8. If not, crop it using torchvision.transforms.CenterCrop
  old_height = photo.shape[1]
  old_width = photo.shape[2]
  new_height = old_height
  new_width = old_width
  radio = old_height / old_width
  max_side = 1000
  if old_height > old_width:
      new_height = max_side
      new_width = int(new_height / radio)
  else:
      new_width = max_side
      new_height = int(new_width * radio)
  if new_width % 8 != 0 or new_height % 8 != 0:
      new_width = new_width // 8 * 8
      new_height = new_height // 8 * 8
  photo = torchvision.transforms.Resize((new_height, new_width))(photo)
  #required_aovs = ["albedo", "normal",]  # "roughness", "metallic", "irradiance"]
  prompts = {
      "albedo": "Albedo (diffuse basecolor)",
      "normal": "Camera-space Normal",
      "roughness": "Roughness",
      "metallic": "Metallicness",
      "irradiance": "Irradiance (diffuse lighting)",
  }
  gen_rgbx = []
  for aov_name in required_aovs:
    generated_image = pipe(
        prompt=prompts[aov_name],
        photo=photo,
        num_inference_steps=num_inference_steps,
        height=new_height,
        width=new_width,
        generator=generator,
        required_aovs=[aov_name],
    ).images[0][0]
    generated_image = torchvision.transforms.Resize(
        (old_height, old_width)
    )(generated_image)
    generated_image = (generated_image, f"Generated {aov_name}")
    gen_rgbx.append(generated_image)
  return gen_rgbx

In [ ]:
lpips_model = lpips.LPIPS(net='vgg').to('cuda')
all_metrics_data = {'albedo': [], 'normal': []}

eval_transform_lpips = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

files = glob('/content/drive/MyDrive/Colab Notebooks/MasterInfo/ADL4CV/FLARE_Dataset/DATA/001/MVI_1812/image/**.png')

for filen in files:
  image_path = f'/content/drive/MyDrive/Colab Notebooks/MasterInfo/ADL4CV/FLARE_Dataset/DATA/001/MVI_1812/image/{filen}'
  path_alpedo = f'/content/drive/MyDrive/Colab Notebooks/MasterInfo/ADL4CV/models_flare/MODELS/001/images_evaluation/qualitative_results/albedo/{filen}'
  path_normal = f'/content/drive/MyDrive/Colab Notebooks/MasterInfo/ADL4CV/models_flare/MODELS/001/images_evaluation/qualitative_results/normal/{filen.zfill(9)}'
  res = rgbx_pipeline(
      pipe,
      image_path,
      20,
      42,
      ["albedo", "normal"]
  )
  for i, pass_ in enumerate(["albedo", "normal"]):
    if pass_ == "albedo":
      gt_image_pil = Image.open(path_alpedo)
    else:
      gt_image_pil = Image.open(path_normal)
    psnr_val, ssim_val, lpips_val = calculate_metrics(
        gen_pil=res[i][0], 
        gt_pil=gt_image_pil,
        lpips_model=lpips_model, 
        lpips_transform=eval_transform_lpips, 
        device='cuda',
   )
    metrics_data = {
        'file': filen,
        'pass': pass_,
        'psnr': psnr_val,
        'ssim': ssim_val,
        'lpips': lpips_val
    }
    print(metrics_data)
    all_metrics_data[pass_].append(metrics_data)

In [ ]:
print("\n" + "="*30)
print(" FINAL EVALUATION METRICS ")
print("="*30)
# In a single-node script, no 'gather' is needed.
all_metrics_list = all_metrics_data['albedo'] + all_metrics_data['normal']
if len(all_metrics_list) == 0:
    print("No metrics gathered.")
else:
    df = pd.DataFrame(all_metrics_list)
    # Calculate and print averages
    for pass_name in ['albedo', 'normal']:
        pass_df = df[df['pass'] == pass_name]
        if pass_df.empty:
            print(f"\nNo metrics calculated for {pass_name}.")
            continue
        avg_psnr = pass_df['psnr'].mean()
        avg_ssim = pass_df['ssim'].mean()
        avg_lpips = pass_df['lpips'].mean()
        print(f"\n--- Average Metrics for: {pass_name} ---")
        print(f"  PSNR:  {avg_psnr:.4f} (Higher is better)")
        print(f"  SSIM:  {avg_ssim:.4f} (Higher is better)")
        print(f"  LPIPS: {avg_lpips:.4f} (Lower is better)")